<a href="https://colab.research.google.com/github/KubiProjekty/orangepi-iot-telemetry-analytics/blob/main/analiza/02_zaawansowany_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import sqlite3

# 1. Pobranie danych bezpośrednio z Twojego repozytorium na GitHubie
host = "raw" + "." + "githubusercontent" + "." + "com"
user = "KubiProjekty"
repo = "orangepi-iot-telemetry-analytics"
base_url = f"https://{host}/{user}/{repo}/main/dane/"

df_loty = pd.read_csv(base_url + "historia_lotow.csv")
df_pogoda = pd.read_csv(base_url + "historia_pogody.csv")
df_smog = pd.read_csv(base_url + "historia_smogu.csv")
df_zdrowie = pd.read_csv(base_url + "historia_zdrowia.csv")

# 2. Czyszczenie i przygotowanie kolumny z pełną godziną pod JOIN w SQL
for df in [df_loty, df_pogoda, df_smog, df_zdrowie]:
    df['Data'] = pd.to_datetime(df['Data'])
    df['Godzina_klucz'] = df['Data'].dt.floor('h').astype(str)

df_loty_czyste = df_loty[df_loty['Predkosc_kmh'] <= 1200].copy()

# 3. Utworzenie bazy SQLite w pamięci i zapisanie tabel
conn = sqlite3.connect(':memory:')
df_loty_czyste.to_sql('loty', conn, index=False, if_exists='replace')
df_pogoda.to_sql('pogoda', conn, index=False, if_exists='replace')
df_smog.to_sql('smog', conn, index=False, if_exists='replace')
df_zdrowie.to_sql('zdrowie', conn, index=False, if_exists='replace')

# 4. Zapytanie SQL nr 1: CASE WHEN + GROUP BY (Ruch lotniczy i pułap wg pór dnia)
query_1 = """
SELECT
    CASE
        WHEN CAST(strftime('%H', Data) AS INTEGER) BETWEEN 6 AND 11 THEN '1. Poranek (06-11)'
        WHEN CAST(strftime('%H', Data) AS INTEGER) BETWEEN 12 AND 17 THEN '2. Popołudnie (12-17)'
        WHEN CAST(strftime('%H', Data) AS INTEGER) BETWEEN 18 AND 22 THEN '3. Wieczór (18-22)'
        ELSE '4. Noc (23-05)'
    END AS pora_dnia,
    COUNT(DISTINCT ICAO24) AS unikalne_samoloty,
    ROUND(AVG(Wysokosc_m), 0) AS srednia_wysokosc_m,
    ROUND(AVG(Predkosc_kmh), 1) AS srednia_predkosc_kmh,
    SUM(CASE WHEN Wysokosc_m < 6000 THEN 1 ELSE 0 END) AS loty_niskie_ponizej_6km
FROM loty
GROUP BY pora_dnia
ORDER BY pora_dnia;
"""

pd.read_sql_query(query_1, conn)


/tmp/ipykernel_1002/1210734682.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Data'] = pd.to_datetime(df['Data'])


,pora_dnia,unikalne_samoloty,srednia_wysokosc_m,srednia_predkosc_kmh,loty_niskie_ponizej_6km
0,1. Poranek (06-11),141,7642.0,670.3,50
1,2. Popołudnie (12-17),154,6741.0,636.0,71
2,3. Wieczór (18-22),160,8020.0,732.3,62
3,4. Noc (23-05),132,10006.0,796.9,19


In [3]:
# ZAPYTANIE 2: CTE (WITH) + Funkcja okna RANK() OVER (PARTITION BY ...)
# Cel: Wyłonienie TOP 3 krajów pochodzenia samolotów dla każdej pory dnia
query_2 = """
WITH statystyki_krajow AS (
    SELECT
        CASE
            WHEN CAST(strftime('%H', Data) AS INTEGER) BETWEEN 6 AND 11 THEN '1. Poranek (06-11)'
            WHEN CAST(strftime('%H', Data) AS INTEGER) BETWEEN 12 AND 17 THEN '2. Popołudnie (12-17)'
            WHEN CAST(strftime('%H', Data) AS INTEGER) BETWEEN 18 AND 22 THEN '3. Wieczór (18-22)'
            ELSE '4. Noc (23-05)'
        END AS pora_dnia,
        Kraj_Pochodzenia,
        COUNT(DISTINCT ICAO24) AS liczba_maszyn
    FROM loty
    GROUP BY pora_dnia, Kraj_Pochodzenia
),
ranking_krajow AS (
    SELECT
        pora_dnia,
        Kraj_Pochodzenia,
        liczba_maszyn,
        RANK() OVER (PARTITION BY pora_dnia ORDER BY liczba_maszyn DESC) AS pozycja_w_rankingu
    FROM statystyki_krajow
)
SELECT *
FROM ranking_krajow
WHERE pozycja_w_rankingu <= 3
ORDER BY pora_dnia, pozycja_w_rankingu;
"""

print("=== TOP 3 KRAJE POCHODZENIA SAMOLOTÓW WG PORY DNIA (CTE + RANK) ===")
display(pd.read_sql_query(query_2, conn))

# POPRAWIONE ZAPYTANIE 3: Najpierw agregujemy pogodę do 1 wiersza na godzinę (CTE), potem robimy JOIN!
query_3_poprawione = """
WITH pogoda_godzinowa AS (
    SELECT
        Godzina_klucz,
        ROUND(AVG(Temp_C), 1) AS srednia_temp_minsk,
        ROUND(AVG(Cisnienie_hPa), 1) AS srednie_cisnienie
    FROM pogoda
    WHERE Miasto LIKE 'Mi%'
    GROUP BY Godzina_klucz
)
SELECT
    z.Data AS czas_pomiaru,
    z.Temp_CPU_C AS cpu_aktualne,
    LAG(z.Temp_CPU_C, 1) OVER (ORDER BY z.Data) AS cpu_poprzednie,
    ROUND(z.Temp_CPU_C - LAG(z.Temp_CPU_C, 1) OVER (ORDER BY z.Data), 2) AS zmiana_cpu,
    ROUND(AVG(z.Temp_CPU_C) OVER (ORDER BY z.Data ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS srednia_kroczaca_3_pomiary,
    p.srednia_temp_minsk AS temp_minsk_maz
FROM zdrowie z
INNER JOIN pogoda_godzinowa p
    ON z.Godzina_klucz = p.Godzina_klucz
ORDER BY z.Data DESC
LIMIT 12;
"""

print("=== POPRAWIONA TELEMETRIA ORANGE PI (ZERO DUPLIKATÓW PO JOIN) ===")
display(pd.read_sql_query(query_3_poprawione, conn))



=== TOP 3 KRAJE POCHODZENIA SAMOLOTÓW WG PORY DNIA (CTE + RANK) ===


,pora_dnia,Kraj_Pochodzenia,liczba_maszyn,pozycja_w_rankingu
0,1. Poranek (06-11),Poland,47,1
1,1. Poranek (06-11),Turkey,22,2
2,1. Poranek (06-11),Hungary,11,3
3,1. Poranek (06-11),Malta,11,3
4,2. Popołudnie (12-17),Poland,76,1
5,2. Popołudnie (12-17),Turkey,13,2
6,2. Popołudnie (12-17),Latvia,5,3
7,2. Popołudnie (12-17),Switzerland,5,3
8,3. Wieczór (18-22),Poland,41,1
9,3. Wieczór (18-22),Turkey,22,2


=== POPRAWIONA TELEMETRIA ORANGE PI (ZERO DUPLIKATÓW PO JOIN) ===


,czas_pomiaru,cpu_aktualne,cpu_poprzednie,zmiana_cpu,srednia_kroczaca_3_pomiary,temp_minsk_maz
0,2026-11-26 18:00:01,41.3,38.0,3.3,40.43,-2.1
1,2026-11-26 17:00:01,38.0,42.0,-4.0,40.83,-2.0
2,2026-11-26 08:00:01,42.0,42.5,-0.5,41.63,-1.8
3,2026-11-26 07:00:01,42.5,40.4,2.1,42.07,-1.0
4,2026-11-26 06:00:02,40.4,43.3,-2.9,41.70,-0.5
5,2026-11-26 05:00:01,43.3,41.4,1.9,42.23,-0.4
6,2026-11-26 04:00:01,41.4,42.0,-0.6,41.60,-0.1
7,2026-11-26 03:00:02,42.0,41.4,0.6,42.70,-0.2
8,2026-11-26 02:00:02,41.4,44.7,-3.3,43.40,0.0
9,2026-11-26 01:00:02,44.7,44.1,0.6,44.23,0.0
